# AIBackends - built-in document tasks with Gemma 4 on llama.cpp

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/AIBackends-llamacpp-document-tasks.ipynb)

Structured extraction, classification, and summarization with local GGUF models through
the `LLAMACPP` runtime. Covers `examples/tasks/`: `basic_task.py` (invoice extraction),
`classify_text.py`, `summarize_text.py`, `extract_custom_schema.py` (your own Pydantic
schema), `sales_call_report.py`, and `video_ad_report.py`.

**Runtime:** works on a CPU runtime; for faster inference pick *Runtime > Change runtime type > T4 GPU*. The device is detected automatically.

In [ ]:
import shutil
import subprocess

# Prebuilt llama-cpp-python wheels: CUDA 12.4 build on GPU runtimes, CPU build otherwise.
HAS_NVIDIA_GPU = (
    shutil.which("nvidia-smi") is not None
    and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
)
LLAMA_WHEEL = (
    "https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.35-cu124/llama_cpp_python-0.3.35-py3-none-manylinux_2_35_x86_64.whl"
    if HAS_NVIDIA_GPU
    else "https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.35/llama_cpp_python-0.3.35-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl"
)
print("GPU runtime:", HAS_NVIDIA_GPU)
print("llama-cpp-python wheel:", LLAMA_WHEEL.rsplit("/", 2)[-2])

%pip install -q "{LLAMA_WHEEL}"
%pip install -q "aibackends>=0.8.1" huggingface_hub

In [2]:
import shutil
import subprocess

import aibackends
import llama_cpp

# "gpu" offloads every layer to CUDA (n_gpu_layers=-1); "cpu" keeps everything on CPU.
HAS_NVIDIA_GPU = (
    shutil.which("nvidia-smi") is not None
    and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
)
DEVICE = "gpu" if HAS_NVIDIA_GPU and llama_cpp.llama_supports_gpu_offload() else "cpu"

print("aibackends", aibackends.__version__)
print("llama-cpp-python", llama_cpp.__version__)
print("device:", DEVICE)

aibackends 0.8.1
llama-cpp-python 0.3.35
device: cpu


In [3]:
from pathlib import Path
from urllib.request import urlretrieve

DATA_URL = "https://raw.githubusercontent.com/donvito/aibackends/main/examples/data"
DATA_DIR = Path("aibackends_data")


def fetch(relative_path: str) -> Path:
    """Download a sample file from the aibackends examples once and return its path."""
    path = DATA_DIR / relative_path
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        partial = path.with_name(path.name + ".part")
        try:
            urlretrieve(f"{DATA_URL}/{relative_path}", partial)
            partial.replace(path)
        finally:
            partial.unlink(missing_ok=True)
    return path

## Model

`GEMMA4_E2B` resolves to `unsloth/gemma-4-E2B-it-GGUF` (Q4_K_M) on the llama.cpp runtime and
is downloaded from the Hugging Face Hub on first use. Passing `device=DEVICE` offloads all
layers to CUDA on a GPU runtime and keeps inference on CPU otherwise.

In [4]:
import json
import time

from aibackends.models import GEMMA4_E2B
from aibackends.runtimes import LLAMACPP
from aibackends.tasks import create_task

LLM = {"runtime": LLAMACPP, "model": GEMMA4_E2B, "device": DEVICE}


def timed(label, fn, *args):
    t = time.perf_counter()
    value = fn(*args)
    print(f"[{label}] {time.perf_counter() - t:.1f}s")
    return value

## 1. Invoice extraction (`ExtractInvoiceTask`)

Returns a validated `InvoiceOutput` Pydantic model. Tasks accept a file path or a string.

In [5]:
from aibackends.tasks import ExtractInvoiceTask

invoice_task = create_task(ExtractInvoiceTask, **LLM)
print(fetch("invoice.txt").read_text()[:400], "...\n")
invoice = timed("invoice", invoice_task.run, fetch("invoice.txt"))
print(invoice.model_dump_json(indent=2))

Invoice
Vendor: Acme Corp
Invoice Number: INV-1042
Invoice Date: 2026-04-10
Bill To: Northwind Logistics

Line Items
- AI integration consulting | qty 1 | unit price 1250.00 | amount 1250.00

Subtotal: 1250.00
Tax: 0.00
Total: 1250.00
Payment Terms: Net 30
 ...



[invoice] 19.8s
{
  "vendor": "Acme Corp",
  "line_items": [
    {
      "description": "AI integration consulting",
      "quantity": 1.0,
      "unit_price": 1250.0,
      "amount": 1250.0
    }
  ],
  "subtotal": 1250.0,
  "tax": 0.0,
  "total": 1250.0,
  "due_date": null,
  "payment_terms": "Net 30"
}


## 2. Document classification (`ClassifyTask`)

In [6]:
from aibackends.tasks import ClassifyTask

classify_task = create_task(
    ClassifyTask,
    labels=["invoice", "rental contract", "employment contract", "receipt", "sales_call"],
    # Naming the schema keys keeps small models from inventing their own field names.
    prompt='Respond with JSON keys "label", "confidence", and "all_scores".',
    **LLM,
)
for doc in ["contract.txt", "invoice.txt", "sales_call.txt"]:
    label = timed(doc, classify_task.run, fetch(doc))
    print("  ->", label.model_dump_json())

[contract.txt] 8.6s
  -> {"label":"rental contract","confidence":0.98,"all_scores":{"invoice":0.05,"rental contract":0.98,"employment contract":0.02,"receipt":0.01,"sales_call":0.01}}


[invoice.txt] 4.7s
  -> {"label":"invoice","confidence":0.99,"all_scores":{"invoice":0.99,"rental contract":0.01,"employment contract":0.0,"receipt":0.0,"sales_call":0.0}}


[sales_call.txt] 4.8s
  -> {"label":"sales_call","confidence":0.95,"all_scores":{"invoice":0.05,"rental contract":0.1,"employment contract":0.1,"receipt":0.05,"sales_call":0.95}}


## 3. Summarization (`SummarizeTask`)

In [7]:
from aibackends.tasks import SummarizeTask

summary = timed("summary", create_task(SummarizeTask, **LLM).run, fetch("meeting_notes.txt"))
print(summary)

[summary] 4.0s
The AI Backends team is preparing for a launch. Key tasks include finalizing the README to emphasize a local-first approach, recording a demo for invoice extraction, publishing a build log on X and LinkedIn, contacting two design partners in the fintech and logistics sectors, and preparing a talk outline for the Singapore AI meetup.
<end_of_turn>


## 4. Custom schema extraction (`ExtractTask`)

Bring any Pydantic model; the runtime constrains output to its JSON schema and validates it.

In [8]:
from pydantic import BaseModel

from aibackends.tasks import ExtractTask


class Lead(BaseModel):
    name: str
    company: str | None = None
    email: str | None = None
    estimated_budget: float | None = None
    priority: str | None = None
    next_step: str | None = None


lead_task = create_task(
    ExtractTask, schema=Lead, instructions="Extract the lead details from the intake note.", **LLM
)
lead = timed("lead", lead_task.run, fetch("lead_note.txt"))
print(lead.model_dump_json(indent=2))

[lead] 5.5s
{
  "name": "Priya Nair",
  "company": "Acme Retail",
  "email": "priya@acmeretail.com",
  "estimated_budget": null,
  "priority": "high",
  "next_step": "send a proposal and timeline by Friday"
}


## 5. Sales-call and video-ad reports

aibackends ships `SalesCallReport` and `VideoAdReport` schemas (used by the
`AnalyseSalesCallTask` / `AnalyseVideoAdTask` built-ins). Small local models follow them
more reliably when the instructions spell out each field's shape, so here we pair the
built-in schemas with `ExtractTask` and explicit field guidance.

In [9]:
from aibackends.schemas.sales_call import SalesCallReport
from aibackends.schemas.video_ad import VideoAdReport

SALES_INSTRUCTIONS = (
    "Analyse the sales call transcript. Return talk_ratio as an object with agent and "
    'customer shares that sum to 1, e.g. {"agent": 0.55, "customer": 0.45}; objections, '
    "buying_signals, and action_items as lists of short strings; score as a number from "
    "0 to 10; sentiment as one word."
)
sales_task = create_task(ExtractTask, schema=SalesCallReport, instructions=SALES_INSTRUCTIONS, **LLM)
report = timed("sales call", sales_task.run, fetch("batch/sales_call_1.txt"))
print(report.model_dump_json(indent=2))

[sales call] 11.7s
{
  "talk_ratio": {
    "agent": 0.55,
    "customer": 0.45
  },
  "objections": [
    "Procurement pilot request"
  ],
  "buying_signals": [
    "Team likes it",
    "Ready to move quickly next month"
  ],
  "action_items": [
    "Propose 50 documents pilot",
    "Set two-week review period"
  ],
  "score": 8.0,
  "sentiment": "Positive"
}


In [10]:
VIDEO_INSTRUCTIONS = (
    "Analyse the video ad brief. Return hook_strength and cta_clarity as numbers from 0 to "
    "10, key_messages as a list of short strings, and emotion_arc as the ordered list of "
    "emotions the ad moves through."
)
video_task = create_task(ExtractTask, schema=VideoAdReport, instructions=VIDEO_INSTRUCTIONS, **LLM)
video = timed("video ad", video_task.run, fetch("video_ad_brief.txt"))
print(video.model_dump_json(indent=2))

[video ad] 5.3s
{
  "hook_strength": 8.0,
  "key_messages": [
    "fresh beans",
    "flexible plans",
    "cancel anytime"
  ],
  "cta_clarity": 9.0,
  "emotion_arc": [
    "curiosity",
    "comfort",
    "urgency"
  ]
}


## 6. From the CLI

The same tasks are available as `aibackends task ...` commands.

In [11]:
!aibackends task summarize --input "$(cat aibackends_data/meeting_notes.txt)" \
    --runtime llamacpp --model gemma4-e2b --device {DEVICE}

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512


The AI Backends team is preparing for a launch by focusing on several key preparatory tasks: finalizing the README to emphasize a local-first approach, recording a demonstration of invoice extraction, publishing a build log on X and LinkedIn, contacting two design partners in the fintech and logistics sectors, and preparing an outline for a meetup talk at Singapore AI.
